In [ ]:
import pandas as pd
import numpy as np
import random

# --- 1. SEGÉDFÜGGVÉNY: SZÖVEGES GENERÁLÁS TORZÍTÁSSAL ---
def generate_descriptive_text(row):
    """
    Szöveges jellemzést generál.
    Képes szimulálni az elfogultságot (bias):
    Ha row['is_biased_opinion'] == 1, akkor a rossz számok ellenére pozitívat ír.
    """

    # --- Alap sablonok ---
    size_hu = {'Micro': 'mikrovállalkozás', 'Small': 'kisvállalkozás', 'Medium': 'középvállalat'}
    entity_hu = row['legal_entity_type']
    industry = row['Industry_code']
    size_str = size_hu.get(row['pl_subseg_desc'], 'cég')

    intro = random.choice([
        f"A(z) {industry} szektorban működő {entity_hu}.",
        f"Jelen ügyfél egy {size_str} ({entity_hu}), fő tevékenysége: {industry}.",
        f"{industry} területen aktív {size_str}."
    ])

    text = intro

    # --- DÖNTÉSI LOGIKA: TORZÍTÁS VS VALÓSÁG ---

    # 1. ESET: ELFOGULT / TORZÍTOTT VÉLEMÉNY (Bias)
    # A számok rosszak, de a szöveg pozitív ("Szépítés")
    if row['is_biased_opinion'] == 1:
        bias_templates = [
            " Bár a pénzügyi mutatók átmeneti gyengeséget mutatnak, a menedzsment tapasztalata garancia a kilábalásra.",
            " A jelenlegi likviditási feszültség kizárólag a piaci környezetnek tudható be, a cég fundamentumai erősek.",
            " Stratégiai ügyfél, hosszú távú együttműködés javasolt a pillanatnyi mutatók ellenére is.",
            " A negatív eredmény egyszeri tételek hatása, a jövőbeli kilátások kifejezetten pozitívak.",
            " Kockázati szempontból elfogadható, a tulajdonosi háttér rendezett, erős piaci pozícióval rendelkezik."
        ]
        text += random.choice(bias_templates)

        # Mentegetőzés a konkrét hibákra
        if row['LatePaymentCount'] > 0:
            text += " A fizetési késedelmek adminisztratív okokra vezethetők vissza."
        elif row['Operating Margin'] < 0:
            text += " A veszteséges működés a beruházási ciklus természetes velejárója."

    # 2. ESET: NORMÁL, ADATALAPÚ VÉLEMÉNY
    else:
        # Pénzügyi helyzet reális értékelése
        if row['loan_grade'] in ['A', 'B']:
            text += random.choice([
                " Kiemelkedő pénzügyi stabilitás és erős tőkehelyzet jellemzi.",
                " A gazdálkodás kiegyensúlyozott, a kockázati profil alacsony.",
                " Megbízható adós, kiváló fedezettségi mutatókkal."
            ])
        elif row['loan_grade'] in ['D', 'E']:
            text += random.choice([
                " A pénzügyi mutatók romló tendenciát és magas kockázatot jeleznek.",
                " A cég tőkehelyzete kritikus, a likviditás feszített.",
                " Magas eladósodottság és alacsony jövedelmezőség jellemzi."
            ])
        else: # C kategória
            text += " Átlagos kockázati besorolású, az iparági átlagnak megfelelő mutatókkal."

        # Konkrét tények (csak ha nem torzított a vélemény)
        details = []
        if row['Operating Margin'] < 0:
            details.append("negatív üzemi eredmény")
        if row['LatePaymentCount'] > 0:
            details.append(f"{row['LatePaymentCount']} db fizetési késedelem")
        if row['has_prior_default'] == 1:
            details.append("korábbi nemfizetés")
        if row['AnnualRevenueGrowth'] > 20:
            details.append("dinamikus növekedés")

        if details:
            text += " Főbb tényezők: " + ", ".join(details) + "."

    # --- Hitelcél és Fedezet (ez objektív marad) ---
    text += f" Hitelcél: {row['loan_purpose'].lower()}."

    if row['property'] == 'Nincs':
        text += " Tárgyi fedezet bevonása nem történt."
    else:
        text += f" Fedezet: {row['property'].lower()} (Érték: {row['collateral_value']:,.0f} HUF)."

    return text


# --- 2. FŐ ADATGENERÁLÓ FÜGGVÉNY ---
def generate_full_dataset_with_bias(n_rows=50000, seed=42):
    print(f"Adatok generálása {n_rows} sorra torzítási logikával... Kérlek várj.")

    np.random.seed(seed)
    random.seed(seed)

    # --- ALAPADATOK ---
    ids = [f"CUST-{100000+i}" for i in range(n_rows)]
    industries = ['Kiskereskedelem', 'Építőipar', 'Gyártás', 'Szállítás', 'IT/Szolgáltatás', 'Mezőgazdaság', 'Vendéglátás', 'Ingatlanügylet']
    subsegs = ['Micro', 'Small', 'Medium']

    industry_col = np.random.choice(industries, n_rows)
    subseg_col = np.random.choice(subsegs, n_rows, p=[0.5, 0.35, 0.15])

    # Jogi forma
    legal_entity = np.empty(n_rows, dtype=object)
    is_micro = subseg_col == 'Micro'
    is_small = subseg_col == 'Small'
    is_medium = subseg_col == 'Medium'

    legal_entity[is_micro] = np.random.choice(['Egyéni Vállalkozó', 'Bt.', 'Kft.'], size=is_micro.sum(), p=[0.4, 0.3, 0.3])
    legal_entity[is_small] = np.random.choice(['Egyéni Vállalkozó', 'Bt.', 'Kft.', 'Zrt.'], size=is_small.sum(), p=[0.05, 0.05, 0.85, 0.05])
    legal_entity[is_medium] = np.random.choice(['Kft.', 'Zrt.'], size=is_medium.sum(), p=[0.7, 0.3])

    # --- PÉNZÜGYI GENERÁLÁS ---
    business_age = np.clip(np.random.lognormal(2.2, 0.8, size=n_rows).astype(int), 1, 100)

    base_sales = np.where(subseg_col == 'Micro', 60_000_000,
                  np.where(subseg_col == 'Small', 600_000_000, 4_000_000_000))
    net_sales = base_sales * np.random.lognormal(0, 0.6, n_rows)

    revenue_per_employee = np.random.uniform(15_000_000, 60_000_000, n_rows)
    num_employees = np.maximum(1, (net_sales / revenue_per_employee).astype(int))

    gross_margin_pct = np.clip(np.random.normal(0.35, 0.12, n_rows), 0.05, 0.95)
    op_margin_pct = gross_margin_pct - np.random.beta(2, 5, n_rows) * 0.45
    ebit = net_sales * op_margin_pct

    asset_turnover = np.random.uniform(0.4, 3.0, n_rows)
    total_assets = net_sales / asset_turnover

    current_assets = total_assets * np.random.uniform(0.2, 0.85, n_rows)

    debt_asset_ratio = np.random.beta(4, 3, n_rows)
    total_liabs = total_assets * debt_asset_ratio
    current_liabs = total_liabs * np.random.uniform(0.3, 1.0, n_rows)

    equity = np.maximum(1000, total_assets - total_liabs)
    retained_earnings = equity * np.random.uniform(0.4, 0.9, n_rows)

    # --- MUTATÓK ---
    safe_current_liabs = np.where(current_liabs == 0, 1, current_liabs)
    current_ratio = current_assets / safe_current_liabs

    inventory_est = current_assets * np.random.uniform(0.3, 0.6, n_rows)
    quick_ratio = (current_assets - inventory_est) / safe_current_liabs

    roa = ebit / total_assets
    roe = ebit / equity
    debt_to_equity = total_liabs / equity

    dso = (current_assets * np.random.uniform(0.2, 0.5, n_rows) / net_sales) * 365

    ocf = ebit + (total_assets * 0.06)
    ocf_ratio = ocf / safe_current_liabs
    ev_revenue = (equity + total_liabs - (current_assets * 0.1)) / net_sales
    rev_growth = np.random.normal(0.06, 0.18, n_rows)

    # Add the working capital calculation here
    working_capital = current_assets - current_liabs

    # --- SZEMÉLYES ÉS HITEL ADATOK ---
    person_age = np.random.randint(23, 78, n_rows)
    present_res_since = np.array([np.random.randint(1, max(2, age - 18)) for age in person_age])

    base_income = np.where(subseg_col == 'Micro', 4_500_000,
                   np.where(subseg_col == 'Small', 14_000_000, 35_000_000))
    person_income = base_income * np.random.lognormal(0, 0.45, n_rows)

    counties = ['Pest', 'Baranya', 'Bács-Kiskun', 'Békés', 'Borsod-Abaúj-Zemplén', 'Csongrád-Csanád', 'Fejér', 'Győr-Moson-Sopron', 'Hajdú-Bihar', 'Heves', 'Jász-Nagykun-Szolnok', 'Komárom-Esztergom', 'Nógrád', 'Somogy', 'Szabolcs-Szatmár-Bereg', 'Tolna', 'Vas', 'Veszprém', 'Zala', 'Budapest']
    address_county = np.random.choice(counties, n_rows)

    properties = ['Ingatlan', 'Gépjármű', 'Nincs', 'Értékpapír', 'Készfizető kezesség', 'Gépsor/Berendezés']
    property_col = np.random.choice(properties, n_rows, p=[0.35, 0.15, 0.25, 0.05, 0.1, 0.1])

    purposes = ['Forgóeszköz finanszírozás', 'Beruházás', 'Eszközbeszerzés', 'Refinanszírozás', 'Likviditási hitel']
    loan_purpose = np.random.choice(purposes, n_rows, p=[0.3, 0.25, 0.2, 0.1, 0.15])

    credit_amount = net_sales * np.random.uniform(0.08, 0.30, n_rows)

    collateral_value = np.zeros(n_rows)
    has_collateral = property_col != 'Nincs'
    collateral_mult = np.random.uniform(1.1, 2.5, size=has_collateral.sum())
    collateral_value[has_collateral] = credit_amount[has_collateral] * collateral_mult

    default_prob_base = 0.05
    risk_adder = np.where(debt_to_equity > 3, 0.05, 0) + np.where(current_ratio < 1, 0.05, 0)
    has_prior_default = np.random.binomial(1, np.clip(default_prob_base + risk_adder, 0, 1), n_rows)

    # --- SCORE ÉS GRADE ---
    risk_factor = (debt_to_equity * 0.12) - (op_margin_pct * 2.5) + (has_prior_default * 2.0)
    late_payment_count = np.random.poisson(np.clip(np.exp(risk_factor), 0, 12), n_rows)

    score = (roa * 12) + (current_ratio * 4) - (debt_to_equity * 0.6) - (late_payment_count * 2.5) - (has_prior_default * 5)
    loan_grade = pd.qcut(score, q=[0, 0.10, 0.25, 0.55, 0.85, 1.0], labels=['E', 'D', 'C', 'B', 'A'])

    affiliate_rev_ratio = np.random.beta(1, 12, n_rows)
    geo_diversification = np.random.beta(1, 6, n_rows) * 0.6

    # --- 3. TORZÍTÁS (BIAS) INJEKTÁLÁSA ---
    # Alapértelmezésben senki sem torzított
    is_biased_opinion = np.zeros(n_rows, dtype=int)

    # KIVÁLASZTJUK A "ROSSZ" ADÓSOKAT (D és E kategória)
    # A pandas Series-t numpy tömbbé konvertáljuk az összehasonlításhoz, ha szükséges,
    # de a qcut kategóriákkal óvatosan kell bánni. Használjuk a string kódokat.
    grade_codes = loan_grade.astype(str)
    bad_debtors_mask = (grade_codes == 'D') | (grade_codes == 'E')

    # Véletlenszerűen kiválasztunk a rossz adósok közül 5%-ot, akiket "szépíteni" fogunk
    # 0.05 valószínűséggel lesz True, de CSAK ott, ahol a mask már True
    random_noise = np.random.rand(n_rows)
    # Feltétel: Rossz adós ÉS a véletlen szám < 0.05
    bias_condition = bad_debtors_mask & (random_noise < 0.05)

    is_biased_opinion[bias_condition] = 1

    # --- DATAFRAME ÖSSZEÁLLÍTÁSA ---
    df = pd.DataFrame({
        'ID': ids,
        'credit_amount': np.round(credit_amount, 2),
        'loan_purpose': loan_purpose,
        'legal_entity_type': legal_entity,
        'num_employees': num_employees,
        'present_res_since': present_res_since,
        'property': property_col,
        'collateral_value': np.round(collateral_value, 2),
        'has_prior_default': has_prior_default,
        'Operating Margin': np.round(op_margin_pct * 100, 2),
        'Current Ratio': np.round(current_ratio, 2),
        'EV/Revenue': np.round(ev_revenue, 2),
        'Return on Assets (ROA)': np.round(roa * 100, 2),
        'Return on Equity (ROE)': np.round(roe * 100, 2),
        'person_age': person_age,
        'person_income': np.round(person_income, 2),
        'loan_grade': loan_grade,
        'MDNA': np.round(debt_to_equity, 2), # MDNA proxy
        'Industry_code': industry_col,
        'TotalAssets': np.round(total_assets, 2),
        'CurrentLiabs': np.round(current_liabs, 2),
        'TotalLiabs': np.round(total_liabs, 2),
        'RetainedEarnings': np.round(retained_earnings, 2),
        'CurrentAssets': np.round(current_assets, 2),
        'NetSales': np.round(net_sales, 2),
        'EBIT': np.round(ebit, 2),
        'DebtToEquityRatio': np.round(debt_to_equity, 2),
        'BusinessAge': business_age,
        'DaysSalesOutstanding (DSO)': np.round(dso, 2),
        'GrossMargin': np.round(gross_margin_pct * 100, 2),
        'AnnualRevenueGrowth': np.round(rev_growth * 100, 2),
        'OperatingCashFlowRatio': np.round(ocf_ratio, 2),
        'QuickRatio': np.round(quick_ratio, 2),
        'WorkingCapital': np.round(working_capital, 2),
        'AffiliateRevenueRatio': np.round(affiliate_rev_ratio * 100, 2),
        'GeographicDiversification': np.round(geo_diversification * 100, 2),
        'LatePaymentCount': late_payment_count,
        'pl_subseg_desc': subseg_col,
        'address_county': address_county,
        'is_biased_opinion': is_biased_opinion # Célváltozó a Bias detektáláshoz!
    })

    # --- SZÖVEG GENERÁLÁSA (MOST MÁR A DF ADATAI ALAPJÁN) ---
    print("Szöveges jellemzések és torzítások generálása...")
    df['description'] = df.apply(generate_descriptive_text, axis=1)

    return df

# --- FUTTATÁS ---
if __name__ == "__main__":
    df_biased = generate_full_dataset_with_bias(n_rows=50000)

    # CSV Mentés
    df_biased.to_csv('synthetic_credit_data_with_bias.csv', index=False)

    # --- ELLENŐRZÉS: Mutassunk egy példát a "Hazugságra" ---
    # Keresünk egy sort, ahol a minősítés rossz (D vagy E), de van torzítás (is_biased_opinion=1)
    biased_example = df_biased[(df_biased['is_biased_opinion'] == 1) & (df_biased['loan_grade'].isin(['D', 'E']))].head(1)

    if not biased_example.empty:
        print("\n--- PÉLDA TORZÍTOTT ADATRA (ELLENTMONDÁS) ---")
        print(f"ID: {biased_example['ID'].values[0]}")
        print(f"Valós Minősítés (Grade): {biased_example['loan_grade'].values[0]} (Ez rossz!)")
        print(f"Profitabilitás (Margin): {biased_example['Operating Margin'].values[0]}%")
        print(f"Késedelmek száma: {biased_example['LatePaymentCount'].values[0]}")
        print(f"Elfogultság Flag (Label): {biased_example['is_biased_opinion'].values[0]}")
        print("\nGenerált (szépített) Szöveg:")
        print(f"\"{biased_example['description'].values[0]}\"")
        print("---------------------------------------------")
    else:
        print("Nem találtam példát a mintában (futtasd újra vagy növeld a mintát).")

Adatok generálása 50000 sorra torzítási logikával... Kérlek várj.
Szöveges jellemzések és torzítások generálása...

--- PÉLDA TORZÍTOTT ADATRA (ELLENTMONDÁS) ---
ID: CUST-100171
Valós Minősítés (Grade): D (Ez rossz!)
Profitabilitás (Margin): 17.32%
Késedelmek száma: 4
Elfogultság Flag (Label): 1

Generált (szépített) Szöveg:
"IT/Szolgáltatás területen aktív mikrovállalkozás. A negatív eredmény egyszeri tételek hatása, a jövőbeli kilátások kifejezetten pozitívak. A fizetési késedelmek adminisztratív okokra vezethetők vissza. Hitelcél: forgóeszköz finanszírozás. Fedezet: ingatlan (Érték: 8,662,936 HUF)."
---------------------------------------------
